# DFT Relaxation Diagnostics

This notebook looks at the benchmark-oriented view of the relaxation workflow. The goal is not to prove chemical accuracy; it is to make status, timing, force, and line-search behavior visible across the local-pseudopotential paths.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mlx_atomistic.benchmarks import dft_geometry

The benchmark smoke uses the same public API as the CLI. Keeping this path programmatic makes it useful in tests, notebooks, and later performance comparisons against custom Metal kernels.

In [ ]:
payload = dft_geometry.build_payload(
    grid_shape=(4, 4, 4),
    steps=1,
    systems="gaussian-dimer,gth-h2,upf-si2",
)
[(case["case"], case["status"], case["final_energy"], case["final_max_force"]) for case in payload["cases"]]

The most useful first diagnostics are total wall time and final maximum force. If one path is much slower or has repeated line-search failures, it becomes the next optimization target.

In [ ]:
cases = payload["cases"]
labels = [case["case"] for case in cases]
times = [case["ms_total"] for case in cases]
forces = [case["final_max_force"] for case in cases]

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
axes[0].bar(labels, times)
axes[0].set_ylabel("wall time / ms")
axes[0].set_title("relaxation smoke cost")
axes[0].tick_params(axis="x", rotation=20)
axes[1].bar(labels, forces, color="tab:red")
axes[1].set_ylabel("final max |F| / Ha bohr⁻¹")
axes[1].set_title("force after smoke step")
axes[1].tick_params(axis="x", rotation=20);

Line-search telemetry is stored per accepted step. In larger systems this is where we would diagnose unstable SCF continuation, too-large trial steps, or noisy force directions.

In [ ]:
line_search = []
for case in cases:
    for step in case["history"]:
        line_search.append(
            {
                "case": case["case"],
                "step": step["index"],
                "accepted_step_size": step["accepted_step_size"],
                "line_search_iterations": step["line_search_iterations"],
                "scf_status": step["scf_status"],
            }
        )
line_search